<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Newton's Method for the Clock Settings

**Use the change in slope to choose a correction direction.**

Keep the four clock observations, corrected-error calculation, and squared-error objective. Remove the calibration equality and correction bound; the settings are unrestricted unless the rate is explicitly held at zero. An offset-only calculation explains curvature; the main calculation lets both settings vary and solves a two-by-two linear system.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · Keep the decision and objective fixed

The observed clock errors are $d=(-2,-1,-1,0)$ min at times $t=(0,1,2,3)$ h. For observation $i=1,\ldots,4$, the corrected error is

$$
y_i=d_i+q+rt_i.
$$

The decision $x=(q,r)$ contains the initial correction $q$ in min and rate correction $r$ in min/h. The response vector is $y=\operatorname{Sim}(x)$. The scalar objective is the sum of squared errors:

$$
f(x)=\sum_{i=1}^{4}(d_i+q+rt_i)^2.
$$

We write $f(x)$ for the objective after composing the response calculation with the squared-error score. Its unit is $\mathrm{min}^2$. Every finite pair of real settings is feasible in this formulation. A smaller score therefore gives a better candidate.

This is a smooth, unconstrained least-squares problem and a quadratic program (QP). These names describe the problem form; gradient descent and Newton's method are solution procedures.

In the calculations, coordinates are numerical values in the stated units: $q=1$ means 1 min, and $r=1$ means 1 min/h. Gradient sizes and step sizes refer to this fixed coordinate convention. Changing units changes the coordinate scaling and can require a different step size.

With $r=0$ held fixed, the one-setting objective is

$$
f(q,0)=(q-2)^2+2(q-1)^2+q^2=4(q-1)^2+2.
$$

This identity gives the exact reference $q^\star=1$ min and $f(q^\star,0)=2\ \mathrm{min}^2$. Every numerical candidate can be compared with this exact reference. The observed data and response calculation remain fixed as the candidate changes.


In [ ]:
import numpy as np

# Fixed observations: each error corresponds to the time at the same position.
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])   # h


def simulate_clock(decision):
    correction, rate = np.asarray(decision, dtype=float)
    return OBSERVED_ERRORS + correction + rate * OBSERVATION_TIMES


def objective_function(response):
    return float(np.dot(response, response))


def evaluate_clock(decision):
    decision = np.asarray(decision, dtype=float)
    if decision.shape != (2,) or not np.isfinite(decision).all():
        raise ValueError("Use two finite numerical settings: correction and rate.")
    response = simulate_clock(decision)
    gradient = 2.0 * np.array([response.sum(), OBSERVATION_TIMES @ response])
    return {"decision": decision.copy(), "response": response,
            "feasible": True, "objective": objective_function(response),
            "gradient": gradient,
            "gradient_norm": float(np.linalg.norm(gradient))}


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Newton's method accounts for changing slope

The superscript $k$ counts numerical updates. Gradient descent uses the slope and a chosen step size. **Newton's method** also uses curvature: how the slope changes as the setting changes. For the offset-only objective, the second derivative is 8, giving

$$
q^{(k+1)}=q^{(k)}-
\frac{\partial f/\partial q}{\partial^2 f/\partial q^2}
=q^{(k)}-\frac{8(q^{(k)}-1)}{8}=1.
$$

This update finds the zero of the local linear approximation to the derivative. Here the derivative is already exactly linear, so the approximation is exact. In the figure the derivative line crosses zero at $q=1$. The vertical axis shows the derivative, not the objective value.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/newton_slope.svg" alt="The offset derivative is the line 8 times q minus 1. At q equal to zero its value is minus 8. Dividing by curvature 8 gives a Newton change of plus 1, reaching the zero derivative at correction 1." width="520" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: white;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Use the curvature of both settings

For the two-setting calculation, the gradient collects the partial derivatives:

$$
\nabla f(q,r)=\begin{bmatrix}8q+12r-8\\12q+28r-6\end{bmatrix}.
$$

For two settings, the **Hessian** $B=\nabla^2 f$ is the matrix of second partial derivatives:

$$
B=
\begin{bmatrix}
8&12\\
12&28
\end{bmatrix}.
$$

The diagonal entries describe each setting's own curvature. The off-diagonal 12 records coupling: changing the rate also changes the offset's slope, and changing the offset changes the rate's slope. We use $B$ for this matrix; $H$ remains reserved for the course's score mapping.

Solve for the Newton direction $p^{(k)}$, then update:

$$
Bp^{(k)}=-\nabla f(x^{(k)}),
\qquad x^{(k+1)}=x^{(k)}+p^{(k)}.
$$

At $(0,0)$ the two equations are $8p_q+12p_r=8$ and $12p_q+28p_r=6$. Their solution is $p=(1.9,-0.6)$. The full step reaches $x^{(1)}=(1.9,-0.6)$, whose errors are $(-0.1,0.3,-0.3,0.1)$ min and score is $0.2\ \mathrm{min}^2$.

The initial negative gradient points toward a positive rate correction, whereas the Newton direction points toward a negative one. Curvature coupling changes the direction as well as its size. For comparison, gradient descent uses $x^{(k+1)}=x^{(k)}-\alpha\nabla f(x^{(k)})$ with $\alpha=0.05$. The contours below compare both methods from the same start. Each contour contains decisions with the same squared-error score.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/clock_update_paths.svg" alt="Contours of the same two-setting clock objective show gradient descent from 0,0 taking several alternating steps and a full Newton step going directly to 1.9,-0.6. Initial gradient descent increases both settings; Newton decreases the rate." width="520" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: white;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Compare full and damped steps

The Hessian is constant because this objective is quadratic. Solving the linear system gives the direction directly, without forming a matrix inverse. A positive damping factor $\eta\le1$ permits a partial Newton step $x^{(k+1)}=x^{(k)}+\eta p^{(k)}$. We use the full step $\eta=1$ for the main comparison.

The numerical runs start at $x^{(0)}=(0,0)$. They stop when the gradient norm satisfies $\|\nabla f\|_2\le10^{-6}$ or when 500 updates have been used. The tolerance measures approximate stationarity in the stated coordinates; the update limit only caps computation.


In [ ]:
HESSIAN = 2.0 * np.array([
    [len(OBSERVED_ERRORS), OBSERVATION_TIMES.sum()],
    [OBSERVATION_TIMES.sum(), OBSERVATION_TIMES @ OBSERVATION_TIMES],
])


def run_newton(initial_decision=(0.0, 0.0), damping=1.0,
               gradient_tolerance=1e-6, max_steps=500):
    if not np.isfinite(damping) or not 0 < damping <= 1:
        raise ValueError("Use a damping factor in (0, 1].")
    if gradient_tolerance < 0 or max_steps < 0:
        raise ValueError("Use a nonnegative tolerance and iteration limit.")
    history = [evaluate_clock(initial_decision)]
    for iteration in range(max_steps + 1):
        current = history[-1]
        if current["gradient_norm"] <= gradient_tolerance:
            reason = "Gradient tolerance reached"
            break
        if iteration == max_steps:
            reason = "Iteration limit reached"
            break
        direction = np.linalg.solve(HESSIAN, -current["gradient"])
        history.append(evaluate_clock(current["decision"] + damping * direction))
    return {"result": history[-1], "history": history,
            "updates": len(history) - 1, "stop_reason": reason,
            "damping": damping, "gradient_tolerance": gradient_tolerance}

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

Full and half steps use the same Hessian, observations, and starting decision. Half steps reach the same tolerance gradually.

In [ ]:
newton_run = run_newton()
damped_run = run_newton(damping=0.5)
for name, run in (("Full Newton", newton_run), ("Half Newton", damped_run)):
    result = run["result"]
    print(name, "settings:", result["decision"], "errors:", result["response"])
    print("Score:", result["objective"], "min² | updates:", run["updates"],
          "|", run["stop_reason"])

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

A full step reaches the exact stationary settings because this objective is quadratic and its Hessian is constant and positive definite. Half steps approach those settings gradually. For a general smooth objective the local curvature can change, so a full Newton step need not reach a minimum or even reduce the objective.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 5 · Source material and scope

The supplied `Lecture_9_2_Traditional_Optimization02/9_2.pptx`, slides 4–8, introduces gradient-based methods, steepest descent, Newton's use of the Hessian, and initialization. Its `Example_9_2/hands_on_1.py` supplies a repeated negative-gradient update, a gradient-norm stopping test, and an iteration limit. This notebook derives full and damped Newton updates for the same clock observations.

The source example's clipping operation is omitted because these settings are unrestricted. Plain gradient and Newton updates do not enforce a calibration equality or correction bound; restoring those requirements would require a constraint-aware method and explicit feasibility checks.
